In [2]:
import sempy
import sempy.fabric as fabric
import pandas
from datetime import datetime
from pyspark.sql.types import StringType, StructField, StructType

StatementMeta(, de2d6794-1f93-44d0-8317-0b12f9bbf0b6, 6, Finished, Available, Finished, False)

In [ ]:
%run 2. Parameters

StatementMeta(, de2d6794-1f93-44d0-8317-0b12f9bbf0b6, -1, Cancelled, , Cancelled, True)

In [ ]:
df_roles = fabric.evaluate_dax(dataset=datasetId, workspace=workspaceId, dax_string=dax_query_roles)
display(df_roles)

StatementMeta(, de2d6794-1f93-44d0-8317-0b12f9bbf0b6, -1, Cancelled, , Cancelled, True)

In [ ]:
df_rolememberships = fabric.evaluate_dax(dataset=datasetId, workspace=workspaceId, dax_string=dax_query_rolememberships)
display(df_rolememberships)

StatementMeta(, de2d6794-1f93-44d0-8317-0b12f9bbf0b6, -1, Cancelled, , Cancelled, True)

In [ ]:
df_tablepermissions = fabric.evaluate_dax(dataset=datasetId, workspace=workspaceId, dax_string=dax_query_tablepermissions)
display(df_tablepermissions)

StatementMeta(, de2d6794-1f93-44d0-8317-0b12f9bbf0b6, -1, Cancelled, , Cancelled, True)

In [ ]:
df_tables = fabric.evaluate_dax(dataset=datasetId, workspace=workspaceId, dax_string=dax_query_tables)
display(df_tables)

StatementMeta(, de2d6794-1f93-44d0-8317-0b12f9bbf0b6, -1, Cancelled, , Cancelled, True)

In [ ]:
import pandas as pd

# Step 1 — Select from df_roles
df_roles_clean = df_roles[["[ID]", "[Name]", "[ModelPermission]"]].copy()
df_roles_clean.columns = ["RoleID", "RoleName", "ModelPermission"]

# Step 2 — Select from df_rolememberships
df_members_clean = df_rolememberships[["[RoleID]", "[MemberName]", "[MemberID]", "[IdentityProvider]", "[MemberType]"]].copy()
df_members_clean.columns = ["RoleID", "MemberName", "MemberID", "IdentityProvider", "MemberType"]

# Step 3 — Select from df_tablepermissions
df_permissions_clean = df_tablepermissions[["[RoleID]", "[TableID]", "[FilterExpression]"]].copy()
df_permissions_clean.columns = ["RoleID", "TableID", "FilterExpression"]

# Step 4 — Select from df_tables
df_tables_clean = df_tables[["[ID]", "[Name]"]].copy()
df_tables_clean.columns = ["TableID", "TableName"]

# Step 5 — Merge all together
df_rls_final = df_roles_clean \
    .merge(df_permissions_clean, on="RoleID", how="left") \
    .merge(df_tables_clean, on="TableID", how="left") \
    .merge(df_members_clean, on="RoleID", how="left")

display(df_rls_final)
print(df_rls_final.columns.tolist())
print(df_rls_final.shape)

StatementMeta(, de2d6794-1f93-44d0-8317-0b12f9bbf0b6, -1, Cancelled, , Cancelled, True)

In [ ]:
# ── Add metadata columns ────────────────────────────────────────────
df_rls_final["WorkspaceId"]  = workspaceId
df_rls_final["DatasetId"]    = datasetId
df_rls_final["DatabaseName"] = semantic_model_name

# ── Convert FabricDataFrame to regular pandas DataFrame ─────────────
pandas_df = pd.DataFrame(df_rls_final).copy()

# ── Convert all column names to uppercase ───────────────────────────
pandas_df.columns = pandas_df.columns.str.upper()

# ── Add timestamp column ─────────────────────────────────────────────
pandas_df["DATABASE_TIMESTAMP"] = f"{semantic_model_name} | {datetime.now():%d.%m.%Y %H:%M:%S}"

# ── Determine version number ─────────────────────────────────────────
table_name = "RLS"

if spark.catalog.tableExists(table_name):
    existing_df = spark.read.table(table_name)

    if "DATABASE_VERSION" in existing_df.columns:
        max_version_row = existing_df.filter(
            existing_df.DATABASE_VERSION.startswith(semantic_model_name)
        ).selectExpr("max(cast(regexp_extract(DATABASE_VERSION, 'V(\\\\d+)', 1) as int)) as max_ver").collect()

        max_version = max_version_row[0]['max_ver'] if max_version_row else None
        next_version = (max_version + 1) if max_version is not None else 1
    else:
        next_version = 1

    write_mode = "append"
else:
    next_version = 1
    write_mode = "overwrite"

print(f"Write mode: {write_mode} | Version: V{next_version}")

# ── Add version column ───────────────────────────────────────────────
pandas_df["DATABASE_VERSION"] = f"{semantic_model_name} | V{next_version}"

# ── Fix all columns — convert everything to string ───────────────────
for col in pandas_df.columns:
    pandas_df[col] = pandas_df[col].astype(str).replace('None', None).replace('nan', None)

# ── Build explicit schema ─────────────────────────────────────────────
schema = StructType([
    StructField(col, StringType(), True) for col in pandas_df.columns
])

# ── Write to Delta table ──────────────────────────────────────────────
spark_df = spark.createDataFrame(pandas_df, schema=schema)
spark_df.write.option("mergeSchema", "true").saveAsTable(name=table_name, mode=write_mode)

print(f"Successfully wrote {spark_df.count()} rows to table '{table_name}' with version V{next_version}")
display(pandas_df)

StatementMeta(, de2d6794-1f93-44d0-8317-0b12f9bbf0b6, -1, Cancelled, , Cancelled, True)

In [4]:
df = spark.sql("SELECT * FROM SMD_LH.dbo.rls LIMIT 1000")
display(df)

StatementMeta(, de2d6794-1f93-44d0-8317-0b12f9bbf0b6, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a0a80d34-4f8c-4374-8cee-949b3b758a9e)